# Harta area diagnosis — 2026-10-22

This executed notebook checks the minimized, private 100-identity capture and saved
pipeline evidence. It makes no source requests or database writes. The original
50 UUIDs were refreshed in the pipeline; the additional 50 are diagnostic only.
They are adjacent, age-ordered cohorts, not a random portal-wide sample.

Full replay code, including PostgreSQL read-only baseline loading, is in
`scripts/research/analyze_harta_area.py`. Private input and generated reports must
be present to rerun. No full descriptions, contacts, real publisher IDs or secrets
are embedded in this notebook. Account proxies are not verified agencies.


In [1]:
import json
from pathlib import Path
from collections import Counter
capture = json.loads(Path('.tmp/harta-area-diagnosis/source_diagnostic_private.json').read_text(encoding='utf8'))
summary = json.loads(Path('reports/generated/harta/area_diagnosis/analysis_summary.json').read_text(encoding='utf8'))
rows = capture['records']
assert len(rows) == len({r['source_id'] for r in rows}) == 100
assert Counter(r['cohort'] for r in rows) == {'original': 50, 'additional': 50}
print('Distinct diagnostic identities:', len(rows))
print('Structured living area:', sum(r['property'].get('living_area') is not None for r in rows))
print('Reported surface area:', sum(r['property'].get('surface_area') is not None for r in rows))
print('Alternate area feature keys:', capture['feature_area_keys'])

Distinct diagnostic identities: 100
Structured living area: 1
Reported surface area: 100
Alternate area feature keys: {}


In [2]:
from groundtruth.services.harta_parsing import HartaParsingService
from groundtruth.services.normalization import NormalizationService
from groundtruth.processing.validation import ListingValidator
parser, normalizer, validator = HartaParsingService(), NormalizationService(), ListingValidator()
accepted, resolved, errors = Counter(), Counter(), Counter()
for row in rows:
    p = parser.parse_payload({'property': row['property']}, url='https://harta.info/property/' + row['source_id'])
    n, factors = normalizer.normalize(p)
    codes = {i.code for i in validator.validate(n).issues} | set(p.extra_fields['source_validation_issues'])
    if factors.location_invalid:
        codes.add('invalid_location')
    accepted[row['cohort']] += not codes
    resolved[row['cohort']] += p.area_sqm is not None
    errors.update(codes)
assert accepted == {'original': 3, 'additional': 2}
print('Accepted after unchanged validation:', dict(accepted))
print('Interior/net evidence resolved:', dict(resolved))
print('Remaining missing interior:', errors['harta_missing_interior_area'])

Accepted after unchanged validation: {'original': 3, 'additional': 2}
Interior/net evidence resolved: {'original': 3, 'additional': 2}
Remaining missing interior: 95


In [3]:
for cohort in ['original', 'additional']:
    subset = [r for r in rows if r['cohort'] == cohort]
    dates = [r['property']['created_at'][:10] for r in subset]
    print(cohort, 'publication dates', min(dates), 'to', max(dates))
    print('publisher account proxies', len({r['publisher_proxy'] for r in subset}))
    print('empty descriptions', sum(not r['description_present'] for r in subset))
groups = {r['publisher_proxy'] for r in rows}
all_missing = sum(all(r['property'].get('living_area') is None for r in rows if r['publisher_proxy'] == g) for g in groups)
print('Groups with structured interior missing throughout:', all_missing, 'of', len(groups))
for s in summary['segments']:
    if s['dimension'] == 'cohort_format':
        print(s['value'], 'sample', s['sample'], 'accepted', s['accepted'])

original publication dates 2026-08-30 to 2026-10-21
publisher account proxies 21
empty descriptions 3
additional publication dates 2026-07-09 to 2026-08-29
publisher account proxies 19
empty descriptions 26
Groups with structured interior missing throughout: 36 of 37
original/unqualified_area sample 38 accepted 1
original/no_description sample 3 accepted 0
original/description_without_area sample 5 accepted 0
original/explicit_area_concept sample 4 accepted 2
additional/description_without_area sample 13 accepted 0
additional/no_description sample 26 accepted 0
additional/unqualified_area sample 9 accepted 0
additional/explicit_area_concept sample 2 accepted 2


In [4]:
root = Path('reports/generated/harta/area_diagnosis')
def overall(folder):
    report = json.loads((root / folder / 'source_quality_report.json').read_text())
    return next(s for s in report['sources'] if s['segment'] == 'all')
before, after = overall('quality_baseline'), overall('quality_persisted')
for k in ['validity_score','completeness_score','location_precision_score','freshness_score','source_quality_score','historical_reliability_score','location_score','proposed_weight']:
    print(k, before[k], '->', after[k])
assert before['proposed_weight'] == after['proposed_weight'] == 1.0
assert after['historical_reliability_score'] is None and after['location_score'] is None
assert after['unique_source_listing_count'] == 50 and after['accepted_count'] == 3
print('Quality describes latest 50 logical identities, not 101 historical versions.')

validity_score 2.0 -> 6.0
completeness_score 69.625 -> 75.125
location_precision_score 24.2 -> 47.0
freshness_score 93.4 -> 93.4
source_quality_score 46.415 -> 48.815
historical_reliability_score None -> None
location_score None -> None
proposed_weight 1.0 -> 1.0
Quality describes latest 50 logical identities, not 101 historical versions.


In [5]:
# Saved result of the actual research audit query:
# select count(*) total,count(distinct source_listing_id) identities,max(id) max_id
# from raw_listings where source_website='harta'
audit = json.loads((root / 'database_audit.json').read_text())
assert audit['raw_listings']['identities'] == 50
assert audit['raw_duplicate_groups'] == 0
assert audit['valuation_eligible'] == 3 and audit['location_candidates'] == 50
checks = json.loads((root / 'repaired_area_verification_private.json').read_text())
assert len(checks) == 4 and all(c['interior_equal'] and c['area_facts_equal'] for c in checks)
print('Persisted identities:', audit['raw_listings']['identities'])
print('Valuation eligible:', audit['valuation_eligible'])
print('Unverified location matching candidates:', audit['location_candidates'])
print('Independently extracted repaired HTML records:', len(checks))

Persisted identities: 50
Valuation eligible: 3
Unverified location matching candidates: 50
Independently extracted repaired HTML records: 4
